# 음함수 미분 — 제약 속의 변화율

> 미적분 7강 · 음함수 미분

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [음함수 미분 — 제약 속의 변화율](https://mioon1402.github.io/timeseriesdata/calc/C07-implicit.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 음함수란 무엇인가

## 1. 값의 지형도와 등고선

## 2. 균형 조건 하나로 끝내기

## 3. 여러 곡선에서 확인하기

## 4. 사다리 문제 — 맞물려 변하는 변수들

## 5. 보너스: ln x 의 미분을 공짜로

## 6. 파이썬으로 확인하기

**7-1. 원 위의 접선 기울기**

In [ ]:
import numpy as np

R = 5.0
print(f"{'각도°':>7} {'x':>9} {'y':>9} {'-x/y':>12} {'√ 로 풀어서':>14}")
for deg in [30, 53.13, 90, 120, 210]:
    t = np.radians(deg)
    x, y = R*np.cos(t), R*np.sin(t)
    음함수 = -x/y if abs(y) > 1e-9 else np.inf
    # y = ±√(R²-x²) 를 직접 미분한 값 (위/아래 반원에 따라 부호가 갈린다)
    부호 = 1 if y >= 0 else -1
    양함수 = 부호 * (-x / np.sqrt(R**2 - x**2)) if abs(x) < R else np.inf
    print(f"{deg:>7} {x:>9.4f} {y:>9.4f} {음함수:>12.5f} {양함수:>14.5f}")

print("\n→ 같은 답이다. 음함수 쪽은 부호를 따로 챙길 필요가 없다.")

**7-2. 일반 공식 −Fx/Fy 를 수치로**

In [ ]:
import numpy as np

def 편미분(F, x, y, 축, h=1e-6):
    if 축 == 'x': return (F(x+h, y) - F(x-h, y)) / (2*h)
    else:         return (F(x, y+h) - F(x, y-h)) / (2*h)

def 기울기(F, x, y):
    Fx, Fy = 편미분(F, x, y, 'x'), 편미분(F, x, y, 'y')
    return -Fx / Fy if abs(Fy) > 1e-12 else np.inf

곡선들 = [
    ("원 x²+y²=25",       lambda x, y: x**2 + y**2,          (3.0, 4.0),  -3/4),
    ("타원 x²/9+y²/4=1",  lambda x, y: x**2/9 + y**2/4,      (1.6209069176, 1.6829419696), None),
    ("x³+y³=2",           lambda x, y: x**3 + y**3,          (1.0, 1.0),  -1.0),
    ("y = sin(x+y)",      lambda x, y: np.sin(x+y) - y,      None,        None),
]

for 이름, F, 점, 손계산 in 곡선들[:3]:
    x, y = 점
    수치 = 기울기(F, x, y)
    print(f"{이름:<20} 점 ({x:.4f}, {y:.4f})   dy/dx = {수치:>10.6f}"
          + (f"   손계산 {손계산}" if 손계산 is not None else ""))

**7-3. y로 풀 수 없는 곡선 — y = sin(x+y)**

In [ ]:
from scipy.optimize import brentq
import numpy as np

# y 를 대수적으로 풀 수는 없다. 수치적으로 찾아야 한다.
def y_of(x):
    return brentq(lambda y: np.sin(x + y) - y, -1.2, 1.2)

print(f"{'x':>7} {'y (수치해)':>14} {'음함수 미분':>14} {'수치 기울기':>14}")
for x in [0.0, 0.5, 1.0, 2.0, 3.0]:
    y = y_of(x)
    # F(x,y) = sin(x+y) - y = 0 이므로  Fx = cos(x+y),  Fy = cos(x+y) - 1
    기울기 = -np.cos(x+y) / (np.cos(x+y) - 1)
    수치 = (y_of(x+1e-5) - y_of(x-1e-5)) / 2e-5
    print(f"{x:>7} {y:>14.8f} {기울기:>14.8f} {수치:>14.8f}")

print("\n→ y 를 '풀지 않고도' 기울기는 정확히 나온다. 이것이 음함수 미분의 실용적 가치다.")

**7-4. 사다리 문제**

In [ ]:
import numpy as np

L = 5.0          # 사다리 길이
dxdt = 0.5       # 바닥이 미끄러지는 속도 (m/s)

print(f"{'x (m)':>8} {'y (m)':>9} {'dy/dt (m/s)':>14} {'속도 비':>10}")
for x in [1, 2, 3, 4, 4.5, 4.9, 4.99]:
    y = np.sqrt(L**2 - x**2)
    dydt = -(x * dxdt) / y            # 2x·dx/dt + 2y·dy/dt = 0
    print(f"{x:>8} {y:>9.4f} {dydt:>14.4f} {abs(dydt/dxdt):>10.3f}")

print("\n→ x=3 에서 dy/dt = -0.375 m/s (교과서의 그 답)")
print("  사다리가 누울수록(y→0) 꼭대기 속도가 폭발한다. 마지막에 '쾅' 떨어지는 이유다.")

**7-5. ln 과 √ 를 음함수로 유도하기**

In [ ]:
import numpy as np

def 수치미분(f, x, h=1e-6):
    return (f(x+h) - f(x-h)) / (2*h)

print("ln x :  e^y = x  ⟹  e^y·dy = dx  ⟹  x·dy = dx  ⟹  y' = 1/x")
for x in [0.5, 1, 2, 10, 100]:
    print(f"  x={x:>6}  수치미분 {수치미분(np.log, x):>12.8f}   1/x = {1/x:>12.8f}")

print("\n√x :  y² = x  ⟹  2y·dy = dx  ⟹  y' = 1/(2y) = 1/(2√x)")
for x in [0.25, 1, 4, 9]:
    print(f"  x={x:>6}  수치미분 {수치미분(np.sqrt, x):>12.8f}   1/(2√x) = {1/(2*np.sqrt(x)):>12.8f}")

**7-6. 연습문제**

In [ ]:
# 문제 1. x³ + y³ = 2 위의 점 (1,1) 에서 dy/dx 를 손으로 구하고 7-2 셀로 확인하세요.

# 문제 2. 반지름 3인 원 x²+y²=9 위에서 접선이 '수평'이 되는 점은 어디일까요?
#         (힌트: dy/dx = -x/y = 0)

# 문제 3. 풍선에 공기를 넣어 반지름이 초당 0.5cm 로 커집니다.
#         반지름이 10cm 인 순간 부피는 초당 얼마나 늘까요?
#         (V = (4/3)πr³ 를 t 로 미분하세요)

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
import numpy as np

# 문제 1 — 3x²dx + 3y²dy = 0 ⟹ dy/dx = -x²/y² = -1
F1 = lambda x, y: x**3 + y**3
print(f"문제 1: 손계산 -1²/1² = -1,  수치 {기울기(F1, 1.0, 1.0):.8f}\n")

# 문제 2 — dy/dx = -x/y = 0 이려면 x=0. 점 (0, ±3)
print("문제 2: -x/y = 0 ⟹ x = 0 ⟹ 점 (0, 3) 과 (0, -3)")
print("        원의 맨 위와 맨 아래. 접선이 수평인 곳이다.")
print(f"        확인: (0,3) 에서 기울기 = {기울기(lambda x,y: x**2+y**2, 1e-9, 3.0):.6f}\n")

# 문제 3 — dV/dt = 4πr²·dr/dt
r, drdt = 10.0, 0.5
print(f"문제 3: dV/dt = 4πr²·dr/dt = {4*np.pi*r**2*drdt:.4f} cm³/s")
print("        4πr² 은 겉넓이다 — 4강에서 본 '경계의 크기' 가 여기서도 나온다.")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)